# Analisis Risiko Banjir Spatio-Temporal Banjarmasin (BARITO)
Notebook ini digunakan untuk keperluan riset, eksplorasi data, dan evaluasi model Hybrid RF-LSTM untuk proyek BARITO.

**Petunjuk Colab:**
1. Upload file `BARITO_PROJECT.zip` ke folder file di samping kiri.
2. Jalankan sel pertama untuk mengekstrak dan memuat modul.

In [ ]:
import os
import sys

# --- OTOMATISASI UNTUK GOOGLE COLAB ---
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('backend'):
        if os.path.exists('BARITO_PROJECT.zip'):
            print("📦 Mengekstrak BARITO_PROJECT.zip...")
            !unzip -q BARITO_PROJECT.zip
        else:
            print("⚠️ File BARITO_PROJECT.zip tidak ditemukan! Silakan upload file tersebut ke folder content Colab.")

# Tambahkan folder backend ke path agar modul bisa diimport
sys.path.append(os.path.join(os.getcwd(), 'backend'))

try:
    from data import generate_spatiotemporal_dataset, RISK_LABELS
    from model import get_model
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    print("✅ Modul BARITO berhasil dimuat!")
except ImportError as e:
    print(f"❌ Gagal memuat modul: {e}")
    print("Tips: Pastikan folder 'backend' ada di direktori yang sama dengan notebook ini.")

## 1. Penarikan Dataset
Menghasilkan data historis selama 12 tahun (2015-2026).

In [ ]:
X_spatial, X_temporal, y, metadata = generate_spatiotemporal_dataset(years=12)
df = pd.DataFrame(metadata)

# Tambahkan fitur untuk analisis
df['elevasi'] = [X_spatial[i][0] for i in range(len(X_spatial))]
df['curah_hujan_bulanan'] = [X_temporal[i][2][1] for i in range(len(X_temporal))]
df['pasang_maks'] = [X_temporal[i][2][2] for i in range(len(X_temporal))]

print(f"Total Data: {len(df)} baris")
df.head()

## 2. Visualisasi Tren Risiko Bulanan

In [ ]:
plt.figure(figsize=(12, 6))
monthly_risk = df.groupby('month')['temporal_risk'].mean()
months_label = ['Jan', 'Feb', 'Mar', 'Apr', 'Mei', 'Jun', 'Jul', 'Agu', 'Sep', 'Okt', 'Nov', 'Des']

sns.barplot(x=months_label, y=monthly_risk.values, palette='RdYlGn_r')
plt.title('Rata-rata Tingkat Risiko Banjir per Bulan (2015-2026)', fontsize=14)
plt.ylabel('Skala Risiko (1-4)')
plt.ylim(1, 4)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

## 3. Evaluasi Model Hybrid RF-LSTM

In [ ]:
model = get_model()
eval_results = model.get_evaluation()

print("=== METRIK EVALUASI MODEL ===")
print(f"Accuracy: {eval_results['accuracy']:.4f}")
print(f"CV Accuracy: {eval_results['cv_mean']:.4f} (+/- {eval_results['cv_std']:.4f})")

print("\n=== PER CLASS REPORT ===")
for cls, metrics in eval_results['per_class'].items():
    print(f"{metrics['label']}: F1-Score = {metrics['f1']:.4f}")